In [ ]:
# OTHELLO bootstrap: make the package importable from notebooks/
import sys
from pathlib import Path
_repo_root = Path.cwd().parent.resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [17]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

# os.environ["TRANSFORMERS_CACHE"] = "/content/drive/Shareddrives/Algoverse_KSAC/hf_cache" #stores model
os.environ["HF_HOME"] = "../hf_home"  # stores logins

hf_token = os.getenv('HF_TOKEN')
# print(type(hf_token))
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [18]:
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path

variants = ['small', 'base', 'large']

paths = {variant:f'../models/google/flan-t5-{variant}' for variant in variants}

model_var = 'large'
current_path =paths[model_var]
save_path = Path(current_path).resolve()

tokenizer = AutoTokenizer.from_pretrained(
    save_path,
    local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    save_path,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True
)

print("Reloaded model successfully")
print(f"model.device = {model.device}")

Reloaded model successfully
model.device = mps:0


#Dataloading (TODO)
#after sparqlgen + name parsing

In [19]:
import pandas as pd

In [20]:
names = ['mintaka', 'hotpot', 'qald']

dataframes = {name:pd.read_csv(f'../data/processed_names/{name}_processed.csv') for name in names}





In [21]:
# from transformers import pipeline
from transformers.generation.utils import GenerationMixin

In [22]:
def answer(question):
    # Simpler, more direct prompt for FLAN
    prompt = f"""

Question: {question}

Answer the given question to the best of your knowledge. Provide a succint and clear short answer."""
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        temperature = 0,
        early_stopping=True,
        do_sample=False,
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    return generated_text

answered = {}
for name, data in dataframes.items():
    print(f'Starting processing for {name}')
    answers = []
    for index, row in data.iterrows():
        question = row['AAVE Question']
        llmanswer = answer(question)
        answers.append(llmanswer)
        print(f'Row {index+1} processed', llmanswer)
    data['Answer'] = answers
    answered[name] = data

Starting processing for mintaka
Row 1 processed Mount Rainier
Row 2 processed James Garner
Row 3 processed James Spader
Row 4 processed 2011
Row 5 processed Donald Trump
Row 6 processed John F. Kennedy
Row 7 processed The Lincoln Memorial was completed in 1865.
Row 8 processed No, Bernie Sanders has never been president.
Row 9 processed Edward Edward Cullen
Row 10 processed 18
Row 11 processed Mississippi River
Row 12 processed 18
Row 13 processed New Jersey
Row 14 processed Lake Baikal
Row 15 processed three
Row 16 processed Los Angeles
Row 17 processed November 30, 2010
Row 18 processed Super Mario Bros.
Row 19 processed Super Mario Bros.
Row 20 processed no
Row 21 processed twice
Row 22 processed Juan Manuel Santos
Row 23 processed Michael Phelps
Row 24 processed no
Row 25 processed John Kasich
Row 26 processed AAVE may refer to:
Row 27 processed Serena Williams was born in 1977.
Row 28 processed Barry Bonds
Row 29 processed Rubber Soul was released in 1967. Revolver was released in

In [23]:
import ast
import re

def clean_text(text):
    # If the entry is actually a list, flatten it
    if isinstance(text, list):
        text = text[0]
    elif isinstance(text, str):
       #If looks like a list []
        if text.strip().startswith('[') and text.strip().endswith(']'):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, list) and len(parsed) > 0:
                    text = parsed[0]
            except Exception:
                pass

    # Now clean as before
    if not isinstance(text, str):
        return text

    text = re.sub(r'^(AAVE|SAE)\s*Question[:\s-]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[\s\[\]\'"]+|[\s\[\]\'"]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text



for name, data in answered.items():
# Apply to both columns
  data['SAE Question'] = data['SAE Question'].apply(clean_text)
  data['AAVE Question'] = data['AAVE Question'].apply(clean_text)
  file_path = f'../data/llm_answers/vanilla-flan-t5-{model_var}/Vanilla_LLM_Answers_{name}.csv'
  data.to_csv(file_path, index=False)
  print(f"Translations completed and saved to {file_path}")





Translations completed and saved to ./LLM_answers/vanilla-flan-t5-large/Vanilla_LLM_Answers_mintaka.csv
Translations completed and saved to ./LLM_answers/vanilla-flan-t5-large/Vanilla_LLM_Answers_hotpot.csv
Translations completed and saved to ./LLM_answers/vanilla-flan-t5-large/Vanilla_LLM_Answers_qald.csv
